Model Setup and Economic Motivation

We study how BDC market returns relate to NAV while explicitly accounting for the fact that NAV is stale and only updated quarterly. Rather than treating NAV as contemporaneous, we use lagged NAV returns, interpreting NAV as the last observable fundamental signal available to investors at time t. This directly addresses the concern that NAV does not reflect real-time information.

Our approach allows the NAV to price relationship to be state-dependent. Specifically, we examine whether macro conditions—captured by changes in the 10-year Treasury rate (discount rate) and credit spreads (risk premia)—affect how much investors rely on NAV when pricing BDCs.

The interaction terms are central. They test whether the explanatory power of NAV declines when valuation is driven more by discounting and risk premia rather than underlying (and stale) fundamentals.


Does NAVs  explanatory power declines when macro conditions shift investors toward discounting and risk premia instead of stale fundamentals.


![image.png](attachment:image.png)



In [1]:
import wrds
import pandas as pd
import numpy as np
import duckdb as dd
import statsmodels.formula.api as smf
db = wrds.Connection()

Loading library list...
Done


In [2]:
df_event_study = pd.read_csv(
    "data/df_event_study.csv",
    parse_dates=["trade_date", "announcement_date"]
).copy()
df_event_study = df_event_study.rename(
    columns={"announcement_date": "event_date", "days_from_event": "event_day"}
)
df_event_study = df_event_study.sort_values(["ticker", "event_date", "trade_date"]).copy()
df_event_study.head()

,ticker,event_date,trade_date,nav,nav_ret,ret,price,event_day
154,ARCC,2016-02-24,2016-02-22,16.457393,-0.019785,0.006275,12.83,-2
155,ARCC,2016-02-24,2016-02-23,16.457393,-0.019785,0.007015,12.92,-1
156,ARCC,2016-02-24,2016-02-24,16.457393,-0.019785,0.000000,12.92,0
157,ARCC,2016-02-24,2016-02-25,16.457393,-0.019785,0.036378,13.39,1
158,ARCC,2016-02-24,2016-02-26,16.457393,-0.019785,0.008962,13.51,2


In [10]:
dd.query("""select ticker, count(*) from df_event_study group by ticker""").to_df()

,ticker,count_star()
0,ARCC,179
1,MAIN,175


We analysize df_trade_nav

Interpretation:

NAV is stale and it does not explain return which is what we expected.
    We could include only 

In [3]:
# event day study regression (using ret)

import statsmodels.api as sm
import pandas as pd

event_days = df_event_study.groupby(["ticker", "event_date"])["event_day"].agg(list)
complete = event_days[event_days.apply(lambda x: sorted(x) == [-1, 0, 1, 2, 3])].index

df_reg = (
    df_event_study
    .set_index(["ticker", "event_date"])
    .loc[complete]
    .reset_index()
    .copy()
)

df_reg = pd.get_dummies(df_reg, columns=["event_day"], prefix="day")

X = df_reg[["day_0", "day_1", "day_2", "day_3"]].astype(float)
X = sm.add_constant(X)

y = pd.to_numeric(df_reg["ret"], errors="coerce").astype(float)

valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.002
Method:                 Least Squares   F-statistic:                     1.045
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.390
Time:                        20:06:11   Log-Likelihood:                 205.67
No. Observations:                  80   AIC:                            -401.3
Df Residuals:                      75   BIC:                            -389.4
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0039      0.005      0.819      0.4

In [4]:
# event study using (market return)

import statsmodels.api as sm
import pandas as pd

market_df = pd.read_csv("data/crsp_market_return.csv", parse_dates=["trade_date"])

df_reg_market_return = df_reg.merge(
    market_df[["ticker", "trade_date", "excess_ret"]].drop_duplicates(["ticker", "trade_date"]),
    on=["ticker", "trade_date"],
    how="left"
)

X = df_reg_market_return[["day_0", "day_1", "day_2", "day_3"]].astype(float)
X = sm.add_constant(X)

y = pd.to_numeric(df_reg_market_return["excess_ret"], errors="coerce").astype(float)

valid_idx = y.notna()
X = X.loc[valid_idx]
y = y.loc[valid_idx]

model = sm.OLS(y, X).fit()
print(model.summary())

df_reg.to_csv("data/reg_df.csv", index=False)
df_reg_market_return.to_csv("data/market_return_df.csv", index=False)

# print(df_reg_market_return["excess_ret"].isna().sum())


                            OLS Regression Results                            
Dep. Variable:             excess_ret   R-squared:                       0.096
Model:                            OLS   Adj. R-squared:                  0.048
Method:                 Least Squares   F-statistic:                     2.000
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.103
Time:                        20:06:12   Log-Likelihood:                 217.31
No. Observations:                  80   AIC:                            -424.6
Df Residuals:                      75   BIC:                            -412.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0024      0.004      0.588      0.5

interpretation:
Day -1 return :- Average return on the day before announcement is negligible .07% (Not signnificant)
Day 0 return :- Average return on the day of announcement is .22% (Not significant)
Day 1 return :- Average return jumps by .46% or 46 basis points.
Mean revision :- 0.0%

Is Day 1 return .46% a NAV surprise?
Regress market return (t+1) = nav_ret

Is it the nav_change that caused the jump on return by the .46% i.e. 46 basis points.

In [5]:
# 1.regress return on day 1 nav_ret

df_day1 = df_event_study[df_event_study['event_day'] == 1].copy()

# 2. Define features: The NAV Surprise
X = df_day1['nav_ret'].astype(float)
X = sm.add_constant(X)

# 3. Define target: The Day 1 Return
y = df_day1['ret'].astype(float)

# 4. Run the OLS
model_sensitivity = sm.OLS(y, X, missing='drop').fit()

print(model_sensitivity.summary())

                            OLS Regression Results                            
Dep. Variable:                    ret   R-squared:                       0.024
Model:                            OLS   Adj. R-squared:                  0.010
Method:                 Least Squares   F-statistic:                     1.685
Date:                Tue, 14 Apr 2026   Prob (F-statistic):              0.199
Time:                        20:06:12   Log-Likelihood:                 182.48
No. Observations:                  72   AIC:                            -361.0
Df Residuals:                      70   BIC:                            -356.4
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0056      0.002      2.365      0.0

Interpretation:
    
    1. alpha on day 1 is .63% => book value is does not matter.
    2. NAV is -.14%
    
    We conclude that NAV has no bearing on the price.


In [6]:
# data integrity #

future_matches = df_reg[df_reg['trade_date'] < df_reg['event_date']]
event_counts = df_reg.groupby('ticker')['event_date'].nunique()
nulls = df_reg[['price', 'nav', 'ret', 'nav_ret']].isnull().sum()
dup_rows = df_reg.duplicated(['ticker', 'event_date', 'trade_date']).sum()

print("--- DATA INTEGRITY TEST RESULTS ---")
print(f"Future-event matches: {len(future_matches)}")
print("\nUnique events per Ticker:")
print(event_counts)
print("\nMissing Values in Regression Columns:")
print(nulls)
print("\nDuplicate ticker-event_date-trade_date rows:")
print(dup_rows)

--- DATA INTEGRITY TEST RESULTS ---
Future-event matches: 16

Unique events per Ticker:
ticker
ARCC    16
Name: event_date, dtype: int64

Missing Values in Regression Columns:
price      0
nav        0
ret        0
nav_ret    0
dtype: int64

Duplicate ticker-event_date-trade_date rows:
0




In this sectionm we add 
1. federal reserve 10 year tbill rate
2. credit spread 


In [ ]:
# df_trade_nav_rate_daily ## for spread study

import duckdb as dd

df_trade_nav_daily = pd.read_csv("data/df_trade_nav_daily.csv", parse_dates=["trade_date"])
df_rate_daily = pd.read_csv("data/df_rate_daily.csv", parse_dates=["rate_date"])

query = """
SELECT 
    t.trade_date,
    t.ret,
    t.event_date as announcement_date,
    d.dgs10,
    d.dbaa,
    CAST(d.dbaa AS DECIMAL(10,3)) - CAST(d.dgs10 AS DECIMAL(10,3)) AS spread
FROM df_event_study t
JOIN df_rate_daily d ON t.trade_date = d.rate_date
"""

df_reg = dd.query(query).to_df()


In [23]:
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354 entries, 0 to 353
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   trade_date         354 non-null    datetime64[ns]
 1   ret                354 non-null    float64       
 2   announcement_date  354 non-null    datetime64[ns]
 3   dgs10              354 non-null    float64       
 4   dbaa               354 non-null    float64       
 5   spread             354 non-null    float64       
dtypes: datetime64[ns](2), float64(4)
memory usage: 16.7 KB
